In [1]:
import sys
import os
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')

from lambda_cox import LambdaSA
from utils import get_churn_lastfm_dataset_months, get_targets_and_masks, train_test_split
import yaml
import jax
import jax.numpy as jnp
import haiku as hk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
path = '/Users/mariana/Documents/projects/Huawei/SurvanData/lastfm-dataset-1K'

In [3]:
seqs, ts, cs = get_churn_lastfm_dataset_months(path, 53, use_static_fs=True)

In [4]:
target, h_ws, mask = get_targets_and_masks(seqs, ts, cs, True)

In [5]:
X_train, X_test, y_train, y_test, hws_train, hws_test, \
        m_train, m_test, ts_train, ts_test, cs_train, cs_test = \
train_test_split(seqs, target, h_ws, mask, ts, cs, 32, test_size=0.2)

In [6]:
m_train[ts_train == 5][0][:6, :6]

array([[ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [False, False, False, False, False, False]])

In [8]:
df_events = pd.read_csv(os.path.join(path, 'events.csv'))

In [9]:
df_events.head()

,userid,month,censored,time
0,user_000001,2009-05,0,26
1,user_000002,2009-04,0,38
2,user_000003,2009-05,0,38
3,user_000004,2009-04,0,25
4,user_000005,2009-05,0,32


In [10]:
prof = pd.read_csv(os.path.join(path, 'userid-profile.tsv'), sep='\t')

In [11]:
len(prof)

992

In [21]:
prof.head()

,#id,gender,age,country,registered
0,user_000001,m,25.367133,Japan,"Aug 13, 2006"
1,user_000002,f,25.367133,Peru,"Feb 24, 2006"
2,user_000003,m,22.000000,United States,"Oct 30, 2005"
3,user_000004,f,25.367133,unknown,"Apr 26, 2006"
4,user_000005,m,25.367133,Bulgaria,"Jun 29, 2006"


In [13]:
prof['gender'] = prof.gender.fillna('unknown')

In [18]:
prof['country'] = prof.country.fillna('unknown')

In [20]:
average_age = prof['age'].mean()
prof['age'] = prof['age'].fillna(average_age)

In [27]:
aux = pd.get_dummies(prof, columns=['gender', 'country'])

In [28]:
bool_columns = aux.select_dtypes(include='bool').columns

# Convert boolean columns to float type
aux[bool_columns] = aux[bool_columns].astype(float)

In [30]:
aux = aux.drop(columns=['registered'])

In [11]:
len(aux.columns)

NameError: name 'aux' is not defined

In [33]:
aux.to_csv(os.path.join(path, 'user_static_features.csv'))